# Simple RNN text classification

Train a small RNN to label short movie comments as **positive** or **negative**. This is a learning example with a deliberately tiny dataset.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

Using: cpu


In [2]:
# 1 means positive; 0 means negative
examples = [
    ('movie was great', 1),
    ('movie was good', 1),
    ('film was amazing', 1),
    ('film was excellent', 1),
    ('movie was bad', 0),
    ('movie was awful', 0),
    ('film was boring', 0),
    ('film was terrible', 0),
]


In [4]:
texts, labels = zip(*examples)

In [5]:
texts

('movie was great',
 'movie was good',
 'film was amazing',
 'film was excellent',
 'movie was bad',
 'movie was awful',
 'film was boring',
 'film was terrible')

In [6]:
labels

(1, 1, 1, 1, 0, 0, 0, 0)

In [7]:
words = set()

In [8]:
for text in texts:
    print(text.split())
    print(type(text.split()))
    for word in text.split():
        words.add(word)
        # print(word)

['movie', 'was', 'great']
<class 'list'>
['movie', 'was', 'good']
<class 'list'>
['film', 'was', 'amazing']
<class 'list'>
['film', 'was', 'excellent']
<class 'list'>
['movie', 'was', 'bad']
<class 'list'>
['movie', 'was', 'awful']
<class 'list'>
['film', 'was', 'boring']
<class 'list'>
['film', 'was', 'terrible']
<class 'list'>


In [9]:
words

{'amazing',
 'awful',
 'bad',
 'boring',
 'excellent',
 'film',
 'good',
 'great',
 'movie',
 'terrible',
 'was'}

In [10]:
words_unique=sorted(words)

In [11]:
words_to_ids = {'<unk>':0}

In [12]:
for i , words in enumerate(words_unique,start=1):
    print(f"value of {i} and word {words}")
    words_to_ids[words]=i

value of 1 and word amazing
value of 2 and word awful
value of 3 and word bad
value of 4 and word boring
value of 5 and word excellent
value of 6 and word film
value of 7 and word good
value of 8 and word great
value of 9 and word movie
value of 10 and word terrible
value of 11 and word was


In [13]:
words_to_ids

{'<unk>': 0,
 'amazing': 1,
 'awful': 2,
 'bad': 3,
 'boring': 4,
 'excellent': 5,
 'film': 6,
 'good': 7,
 'great': 8,
 'movie': 9,
 'terrible': 10,
 'was': 11}

In [14]:
def encode(text):
    token = []
    for word in text.split():
        token.append(words_to_ids.get(word,0))
    return token

In [15]:
encode("movie was great")

[9, 11, 8]

In [16]:
texts

('movie was great',
 'movie was good',
 'film was amazing',
 'film was excellent',
 'movie was bad',
 'movie was awful',
 'film was boring',
 'film was terrible')

In [17]:
x_input =[]
for text in texts:
    x_input.append(encode(text))

In [18]:
x_input

[[9, 11, 8],
 [9, 11, 7],
 [6, 11, 1],
 [6, 11, 5],
 [9, 11, 3],
 [9, 11, 2],
 [6, 11, 4],
 [6, 11, 10]]

In [19]:
X=torch.tensor(x_input,dtype=torch.long)

In [20]:
y=torch.tensor(labels,dtype=torch.float32)

In [21]:
print(X,y)

tensor([[ 9, 11,  8],
        [ 9, 11,  7],
        [ 6, 11,  1],
        [ 6, 11,  5],
        [ 9, 11,  3],
        [ 9, 11,  2],
        [ 6, 11,  4],
        [ 6, 11, 10]]) tensor([1., 1., 1., 1., 0., 0., 0., 0.])


In [22]:
class MovieRNN(nn.Module):
    def __init__(self,vocal_size):
        super().__init__()
        self.embedding = nn.Embedding(vocal_size,8)
        self.rnn = nn.RNN(8,16,batch_first=True)
        self.output = nn.Linear(16,1)

    def forward(self,x):
        x = self.embedding(x)
        _,hidden=self.rnn(x)
        return self.output(hidden[-1]).squeeze(1)

In [23]:
movie=MovieRNN(len(words_to_ids))

In [24]:
loss_fn=nn.BCEWithLogitsLoss()

In [25]:
optimizer = torch.optim.Adam(movie.parameters(),lr=0.02)

In [26]:
for epoch in range(200):
    logit=movie(X)
    loss=loss_fn(logit,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch + 1}: loss = {loss.item():.4f}')

Epoch 50: loss = 0.0006
Epoch 100: loss = 0.0003
Epoch 150: loss = 0.0002
Epoch 200: loss = 0.0002


In [27]:
movie.eval()

MovieRNN(
  (embedding): Embedding(12, 8)
  (rnn): RNN(8, 16, batch_first=True)
  (output): Linear(in_features=16, out_features=1, bias=True)
)

In [28]:
text = "movie was awful"

In [31]:
with torch.no_grad():
    x=torch.tensor([encode(text)])
    probability=torch.sigmoid(movie(x)).item()
    if probability>=0.5:
        print("positive")
    else:
        print("negative")



negative


In [36]:
#Task 1
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

# Take a smaller sample for faster training
df = df.sample(n=2500, random_state=42).reset_index(drop=True)

# Display the first 5 rows
print(df.head())

# Display dataset size
print("Dataset size:", df.shape)


                                              review sentiment
0  I really liked this Summerslam due to the look...  positive
1  Not many television shows appeal to quite as m...  positive
2  The film quickly gets to a major chase scene w...  negative
3  Jane Austen would definitely approve of this o...  positive
4  Expectations were somewhat high for me when I ...  negative
Dataset size: (2500, 2)


In [37]:
#Task 2

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())

# Check label counts
print(df["label"].value_counts())


                                              review sentiment  label
0  I really liked this Summerslam due to the look...  positive      1
1  Not many television shows appeal to quite as m...  positive      1
2  The film quickly gets to a major chase scene w...  negative      0
3  Jane Austen would definitely approve of this o...  positive      1
4  Expectations were somewhat high for me when I ...  negative      0
label
1    1261
0    1239
Name: count, dtype: int64


In [39]:
#Task 3
from sklearn.model_selection import train_test_split

# Split reviews and labels
X = df["review"]
y = df["label"]

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check sizes
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 2000
Testing samples: 500


In [40]:
!pip install tiktoken


In [41]:
#Task 4
import tiktoken

# Load the cl100k_base tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

# Tokenize the training and test reviews
X_train_tokens = [encoding.encode(review) for review in X_train]
X_test_tokens = [encoding.encode(review) for review in X_test]

# Display an example
print("Original review:")
print(X_train.iloc[0])

print("\nToken IDs:")
print(X_train_tokens[0])

print("\nNumber of tokens:")
print(len(X_train_tokens[0]))


Original review:
Valley Girl is the definitive 1980's movie with catch phrases filtered throughout this wonderfully acted movie. The characters are so convincing that you forget it is a movie and not a video of an actual "day-in-the-life" of any high school, USA. This flick is to the 1980's what the Brady Bunch TV series is to the 1970's. If you don't like it, well then "Gag me with a spoon."

Token IDs:
[2257, 3258, 11617, 374, 279, 45813, 220, 3753, 15, 596, 5818, 449, 2339, 32847, 18797, 6957, 420, 61085, 31532, 5818, 13, 578, 5885, 527, 779, 40661, 430, 499, 10894, 433, 374, 264, 5818, 323, 539, 264, 2835, 315, 459, 5150, 330, 1316, 3502, 10826, 26928, 1, 315, 904, 1579, 2978, 11, 7427, 13, 1115, 29447, 374, 311, 279, 220, 3753, 15, 596, 1148, 279, 36470, 426, 3265, 6007, 4101, 374, 311, 279, 220, 4468, 15, 596, 13, 1442, 499, 1541, 956, 1093, 433, 11, 1664, 1243, 330, 38, 351, 757, 449, 264, 46605, 1210]

Number of tokens:
94


In [42]:
#task 5
import torch

MAX_LENGTH = 200
PAD_TOKEN = 0

def pad_or_truncate(tokens, max_length=MAX_LENGTH):
    # Truncate
    if len(tokens) > max_length:
        tokens = tokens[:max_length]

    # Pad
    elif len(tokens) < max_length:
        tokens = tokens + [PAD_TOKEN] * (max_length - len(tokens))

    return tokens


# Apply to all tokenized reviews
X_train_padded = [
    pad_or_truncate(tokens)
    for tokens in X_train_tokens
]

X_test_padded = [
    pad_or_truncate(tokens)
    for tokens in X_test_tokens
]

# Convert to tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

print("Training shape:", X_train_tensor.shape)
print("Testing shape:", X_test_tensor.shape)


Training shape: torch.Size([2000, 200])
Testing shape: torch.Size([500, 200])


In [43]:
#task 5
import torch

# Convert token sequences to tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

# Convert labels to tensors
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Check shapes
print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)


X_train: torch.Size([2000, 200])
y_train: torch.Size([2000])
X_test: torch.Size([500, 200])
y_test: torch.Size([500])


In [44]:
#Task 7
import torch
import torch.nn as nn

# Get vocabulary size from tiktoken
VOCAB_SIZE = encoding.n_vocab

class RNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=VOCAB_SIZE,
            embedding_dim=128,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            input_size=128,
            hidden_size=128,
            batch_first=True
        )

        # Linear output layer
        self.linear = nn.Linear(128, 1)

    def forward(self, x):

        # Token IDs -> embeddings
        x = self.embedding(x)

        # Embedding -> RNN
        output, hidden = self.rnn(x)

        # Take the final time step
        x = output[:, -1, :]

        # RNN -> Linear
        x = self.linear(x)

        return x


# Create the model
model = RNNClassifier()

print(model)


RNNClassifier(
  (embedding): Embedding(100277, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (linear): Linear(in_features=128, out_features=1, bias=True)
)


In [45]:
#task 8
import torch
import torch.nn as nn

# Loss function
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Number of epochs
EPOCHS = 5

# Training loop
for epoch in range(EPOCHS):

    model.train()

    # Forward pass
    outputs = model(X_train_tensor)

    # Convert labels to float for BCEWithLogitsLoss
    loss = criterion(
        outputs.squeeze(),
        y_train_tensor.float()
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss
    print(
        f"Epoch [{epoch + 1}/{EPOCHS}], "
        f"Loss: {loss.item():.4f}"
    )


Epoch [1/5], Loss: 0.7001
Epoch [2/5], Loss: 0.6913
Epoch [3/5], Loss: 0.6836
Epoch [4/5], Loss: 0.6768
Epoch [5/5], Loss: 0.6704


In [46]:
# Model evaluation mode
model.eval()

# Disable gradient calculation
with torch.no_grad():

    # Get predictions
    outputs = model(X_test_tensor)

    # Convert logits to probabilities
    probabilities = torch.sigmoid(outputs.squeeze())

    # Convert probabilities to 0 or 1
    predictions = (probabilities >= 0.5).long()

    # Calculate accuracy
    correct = (predictions == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)

    accuracy = correct / total

print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 51.00%


In [47]:
#10
import torch
import tiktoken

# Tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

MAX_LENGTH = 200
PAD_TOKEN = 0

def predict_sentiment(review):
    # Tokenize
    tokens = encoding.encode(review)

    # Truncate
    tokens = tokens[:MAX_LENGTH]

    # Pad
    if len(tokens) < MAX_LENGTH:
        tokens += [PAD_TOKEN] * (MAX_LENGTH - len(tokens))

    # Convert to tensor
    input_tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    )

    # Evaluation mode
    model.eval()

    with torch.no_grad():
        output = model(input_tensor)

        probability = torch.sigmoid(output).item()

        if probability >= 0.5:
            sentiment = "Positive"
        else:
            sentiment = "Negative"

    return sentiment


# 3 new reviews
reviews = [
    "This movie was absolutely fantastic and I loved every minute of it.",
    "The movie was boring, poorly acted, and a complete waste of time.",
    "Amazing story, excellent acting, and a wonderful experience."
]


# Test the model
for review in reviews:
    prediction = predict_sentiment(review)

    print("\nReview:", review)
    print("Predicted Sentiment:", prediction)



Review: This movie was absolutely fantastic and I loved every minute of it.
Predicted Sentiment: Positive

Review: The movie was boring, poorly acted, and a complete waste of time.
Predicted Sentiment: Positive

Review: Amazing story, excellent acting, and a wonderful experience.
Predicted Sentiment: Positive
